# Catboost

Cat boost is a gradient boosting algorithm that uses categorical features. It is a popular choice for handling categorical data in machine learning.

In [1]:
# !pip install catboost -q

In [3]:
pip install --upgrade numpy

  Using cached numpy-2.2.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.1-cp312-cp312-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
catboost 1.2.7 requires numpy<2.0,>=1.16.0, but you have numpy 2.2.1 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


<frozen importlib._bootstrap>:488: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 216 from C header, got 232 from PyObject


In [4]:
pip install --upgrade catboost

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.1
    Uninstalling numpy-2.2.1:
      Successfully uninstalled numpy-2.2.1
Note: you may need to restart the kernel to use updated packages.


In [5]:
# data import
df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


Pre-Processing

In [8]:
df.isnull().sum().sort_values(ascending=False)

deck           688
age            177
embarked         2
embark_town      2
survived         0
pclass           0
sex              0
sibsp            0
parch            0
fare             0
class            0
who              0
adult_male       0
alive            0
alone            0
dtype: int64

In [18]:
# impute missing values using knn imputers in fare, age
from sklearn.impute import KNNImputer
knn_imputer = KNNImputer(n_neighbors=5)

df[['age']] = knn_imputer.fit_transform(df[['age']])

# impute missing values using mode in embarked, embark_town using simple imputer
# from sklearn.impute import SimpleImputer
# simple_imputer = SimpleImputer(strategy='most_frequent')

# df[['embarked', 'embark_town']] = simple_imputer.fit_transform(df[['embarked', 'embark_town']])
# df.head()

In [17]:
# impute embarked missing values using pandas
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])
# drop deck column
df.drop('deck', axis=1, inplace=True)

# df missing values
df.isnull().sum().sort_values(ascending=False)

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [20]:
# convert each category column to category
categorical_columns = df.select_dtypes(include=['object', 'category']).columns

# add this as a new column in a dataframe
df[categorical_columns] = df[categorical_columns].astype('category')

In [21]:
# split data into x and y
x = df.drop('survived', axis=1)
y = df['survived']

# split the data into train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [22]:
# run the catboost classifier

model = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, eval_metric='Accuracy', random_seed=42, verbose=False)

In [31]:
# train the model
model.fit(x_train, y_train, cat_features=categorical_columns.tolist())

# predict the data
y_pred = model.predict(x_test)

# evaluate the model
print(f'accuracy score: ', accuracy_score(y_test, y_pred))
print(f'confusion matrix:\n', confusion_matrix(y_test, y_pred))
print(f'classification report:\n', classification_report(y_test, y_pred))

accuracy score:  1.0
confusion matrix:
 [[105   0]
 [  0  74]]
classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       105
           1       1.00      1.00      1.00        74

    accuracy                           1.00       179
   macro avg       1.00      1.00      1.00       179
weighted avg       1.00      1.00      1.00       179

